In [1]:
import os
from openai import OpenAI
from pyexpat.errors import messages

client = OpenAI(api_key=os.environ['DEEPSEEK_API_KEY'], base_url='https://api.deepseek.com')
system_prompt = """
    用户会对你进行提问,你需要输出结构化json格式,格式如下:
    {
        "question":"用户的问题",
        "answer:"AI的回答"
    }
    question是你对用户问题的提炼,answer是你的简短回答
"""
messages = []
messages.append({"role": "system", "content": f"{system_prompt}"})
messages.append({"role": "user", "content": "福田最好的小学"})
response = client.chat.completions.create(
    model="deepseek-flash",
    messages=messages,
    stream=False,
    reasoning_effort="high",
    extra_body={"thinking": {"type": "enabled"}}
)

print(response.choices[0].message.content)

{
  "question": "福田最好的小学",
  "answer": "没有官方唯一排名。综合口碑第一梯队通常包括：深圳实验学校小学部、荔园小学、园岭小学、东海实验小学、福田区外国语学校小学部、侨香外国语学校小学部、明德实验学校小学部、石厦学校小学部等。若按民间公认度，常被提到前三的是深圳实验学校小学部、荔园小学、园岭小学。择校还要看学区、直升初中、离家距离和教育理念。"
}


In [3]:
def search():
    return True


def read_file(filename):
    """
    :param filename: 文件名
    :return: 返回文件的文本内容
    """
    try:
        with open(filename, "r", encoding="utf-8") as f:
            text = f.read()
    except FileNotFoundError as e:
        return e.filename, e.strerror
    return text

In [5]:
def send_messages(messages):
    response = client.chat.completions.create(
        model="deepseek-flash",
        messages=messages,
        stream=False,
        reasoning_effort="high",
        tools=tools, )
    return response.choices[0].message


tools = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "用户提供TXT格式的文件名,返回文件的文本内容",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {
                        "type": "string",
                        "description": "TXT格式的文件名",
                    }
                },
                "required": ["filename"]
            },
        }
    },
]
messages = []
messages.append({"role": "user", "content": "text.txt这个文件说了什么"})
message = send_messages(messages)
print(f"user>\t{messages[0]['content']}")

tool = message.tool_calls[0]
messages.append(message)
messages.append({"role":"tool","tool_call_id":tool.id,})



user>	ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_xh9YfmHLh924XOvo8LbZ5346', function=Function(arguments='{"filename": "text.txt"}', name='get_weather'), type='function', index=0)], reasoning_content="The user wants to know what's in text.txt. Let me call the tool.")
